## 02. Data Cleaning
Here we clean the dataset. In the prior EDA we found the dataset was very clean overall, so there is not much to do in this area.

## 01. Imports

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TRAIN_RAW_PATH,
    TRAIN_CLEAN_PATH,
    TARGET,
    ID_COLUMN,
)

pd.set_option("display.max_columns", 100)

## 02. Load Data

In [2]:
train = pd.read_csv(TRAIN_RAW_PATH)

print("Shape:", train.shape)
display(train.head())

Shape: (1058, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


## 03. Confirm Main EDA Findings
The EDA found no missing values, duplicate rows, or invalid records. A quick check here confirms that before cleaning.  Useful if dataset changes.

In [3]:
print("Missing values:", train.isna().sum().sum())
print("Duplicate rows:", train.duplicated().sum())
print("Duplicate employee IDs:", train[ID_COLUMN].duplicated().sum())
print("Attrition rate:", f"{train[TARGET].mean():.1%}")

Missing values: 0
Duplicate rows: 0
Duplicate employee IDs: 0
Attrition rate: 16.9%


In [4]:
constant_columns = [
    column for column in train.columns
    if train[column].nunique() == 1
]

constant_columns

['EmployeeCount', 'Over18', 'StandardHours']

## 04. Keep the outliers

The EDA identified several statistical outliers using the IQR rule, particularly in income and tenure-related variables. The values were still realistic for employees, and the consistency checks did not identify invalid records.

For that reason, no rows are removed simply for being statistical outliers.

In [5]:
print("Rows before cleaning:", len(train))
print("Rows removed for outliers: 0")

Rows before cleaning: 1058
Rows removed for outliers: 0


## 5. Drop the constant columns
Dropping these because these were either the same for everyone or nearly the same, thus unlikely to be predictive.  `EmployeeNumber` is left in the cleaned dataset so predictions can later be matched back to the correct employee record. Each modeling notebook will remove it from the feature matrix before training.

In [6]:
columns_to_drop = [
    "EmployeeCount",
    "Over18",
    "StandardHours",
]

train_clean = train.drop(columns=columns_to_drop)

print("Original shape:", train.shape)
print("Cleaned shape:", train_clean.shape)


Original shape: (1058, 35)
Cleaned shape: (1058, 32)


## 6. Sanity Check for Cleaned Dataset

In [7]:
display(train_clean.head())

print("Remaining missing values:", train_clean.isna().sum().sum())
print("Remaining duplicate rows:", train_clean.duplicated().sum())
print("Rows:", train_clean.shape[0])
print("Columns:", train_clean.shape[1])

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2


Remaining missing values: 0
Remaining duplicate rows: 0
Rows: 1058
Columns: 32


## 7. Save the Cleaned Dataset

In [8]:
TRAIN_CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

train_clean.to_csv(TRAIN_CLEAN_PATH, index=False)

print("Saved to:", TRAIN_CLEAN_PATH)

Saved to: C:\Users\jeffh\PycharmProjects\employee-attrition\data\processed\train_clean.csv
